## 1. ⚙️ Cài đặt & Tối ưu Phần cứng GPU

Bật **Mixed Precision (float16)** để tận dụng Tensor Cores của GPU — tốc độ train tăng ~2x mà không mất độ chính xác.
Đặt **seed toàn cục = 42** để tái tạo kết quả hoàn toàn.



In [ ]:
import os, sys, random, json, logging
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, CSVLogger

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

logging.getLogger('tensorflow').setLevel(logging.ERROR)
tf.get_logger().setLevel('ERROR')

# ── Mixed Precision: float16 compute, float32 variables ──────────────────
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print("Mixed precision policy :", policy.name)
print("Compute dtype          :", policy.compute_dtype)
print("Variable dtype         :", policy.variable_dtype)

# ── Global random seeds ───────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ── GPU memory growth (prevent OOM) ──────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"\nGPUs found : {len(gpus)}")
    print(f"Device     : {gpus[0].name}")
else:
    print("WARNING: No GPU detected — training will be very slow!")

print("TensorFlow version:", tf.__version__)



## 2. 📁 Kết nối Google Drive & Định nghĩa Đường dẫn

Mount Drive và cấu hình đường dẫn chuẩn cho toàn bộ project.
Tạo các thư mục output cục bộ để lưu model, log và biểu đồ.



In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [ ]:
# ── Google Drive paths ────────────────────────────────────────────────────
BASE_DIR  = Path('/content/drive/MyDrive/Brain-Tumor-Mri-Deep-Learning')
JSON_PATH = BASE_DIR / 'DATA.json'
IMG_DIR   = BASE_DIR / 'data' / 'raw'

# ── Local output directories ──────────────────────────────────────────────
MODEL_DIR   = Path('./models')
REPORT_DIR  = Path('./reports')
FIGURE_DIR  = REPORT_DIR / 'figures'
LOG_DIR     = Path('./logs')
HEATMAP_DIR = Path('/tmp/heatmaps')       # offline pre-computed heatmaps

for d in [MODEL_DIR, REPORT_DIR, FIGURE_DIR, LOG_DIR, HEATMAP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Hyper-parameters ──────────────────────────────────────────────────────
IMG_SIZE      = 384    # optimal balance: GPU memory vs. high-res medical detail
HEATMAP_SIZE  = 48     # output size of heatmap head (12×12 backbone → 48×48)
HEATMAP_SIGMA = 4      # Gaussian sigma in 48×48 space
BATCH_SIZE    = 16     # adjust if OOM (try 8 for Tesla P100)
EPOCHS_P1     = 10     # Phase 1: warm-up (head only)
EPOCHS_P2     = 50     # Phase 2: fine-tune (top backbone layers)

print(f"BASE_DIR  : {BASE_DIR}")
print(f"JSON_PATH : {JSON_PATH}")
print(f"IMG_DIR   : {IMG_DIR}")
print(f"Image size: {IMG_SIZE}×{IMG_SIZE}")



## 3. 🧠 Tải Dữ liệu & Tạo Heatmap Ngoại tuyến (Offline Pre-computation)

### Tại sao tạo heatmap trước khi train?
Nếu tạo heatmap on-the-fly trong mỗi batch, CPU sẽ trở thành bottleneck và GPU phải chờ.
Bằng cách pre-compute và lưu ra đĩa, CPU bottleneck được loại bỏ hoàn toàn.

### ⚠️ Ngăn chặn Data Leakage (QUAN TRỌNG)
Với các class bắt đầu bằng **"Normal"**, marker X,Y chỉ là placeholder ở **giữa ảnh** — không có tổn thương thật.
→ Heatmap của Normal PHẢI là **mảng toàn số 0** (không có Gaussian).
→ Nếu để Gaussian ở giữa, model sẽ học shortcut sai: "ảnh có activation ở trung tâm = Normal".



In [ ]:
def load_dataset_info(json_path, img_dir):
    """
    Parse DATA.json: resolve full image paths, extract class label
    and lesion center point. Normal classes → point = None.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)

    dataset_info = []
    classes_set  = set()
    missing      = 0

    for rel_path, meta in raw.items():
        safe      = rel_path.replace('\\', os.sep).replace('/', os.sep)
        full_path = Path(img_dir) / safe

        if not full_path.exists():
            missing += 1
            continue

        cls = meta['class']
        classes_set.add(cls)

        # DATA LEAKAGE PREVENTION: Normal has no real lesion
        if cls.startswith('Normal'):
            point = None
        else:
            point = (meta['point']['x'], meta['point']['y'])

        dataset_info.append({
            'img_path' : str(full_path),
            'class'    : cls,
            'point'    : point,
            'width'    : meta.get('width',  512),
            'height'   : meta.get('height', 512),
        })

    classes_list = sorted(classes_set)
    class_to_idx = {c: i for i, c in enumerate(classes_list)}

    print(f"Images loaded   : {len(dataset_info):,}")
    print(f"Missing/skipped : {missing}")
    print(f"Total classes   : {len(classes_list)}")
    return dataset_info, class_to_idx, classes_list


dataset_info, class_to_idx, CLASS_NAMES = load_dataset_info(JSON_PATH, IMG_DIR)
NUM_CLASSES = len(CLASS_NAMES)



In [ ]:
# ── Class distribution ────────────────────────────────────────────────────
from collections import Counter
count_map = Counter(d['class'] for d in dataset_info)
dist_df   = pd.DataFrame(count_map.items(), columns=['class_name','count'])
dist_df   = dist_df.sort_values('class_name').reset_index(drop=True)
display(dist_df)

plt.figure(figsize=(16, 5))
plt.bar(dist_df['class_name'], dist_df['count'], color='steelblue')
plt.title('Full Dataset – Class Distribution (30 classes)')
plt.xlabel('Class'); plt.ylabel('Image Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'class_distribution.png', dpi=150)
plt.show()



In [ ]:
# ── Gaussian heatmap function ─────────────────────────────────────────────
def generate_gaussian_heatmap(size: int, center, sigma: float) -> np.ndarray:
    """
    Generate a 2D Gaussian at `size x size` resolution centred on `center`.

    CRITICAL: If center is None (Normal class), returns a ZERO array —
    this prevents the model from learning a false 'centre = normal' shortcut.
    """
    heatmap = np.zeros((size, size), dtype=np.float32)

    if center is None:              # Normal class — DATA LEAKAGE PREVENTION
        return heatmap

    cx, cy = float(center[0]), float(center[1])

    # Guard: out-of-bounds coordinates
    if not (0 <= cx < size and 0 <= cy < size):
        return heatmap

    x = np.arange(size, dtype=np.float64)
    y = np.arange(size, dtype=np.float64)[:, np.newaxis]
    heatmap = np.exp(-4.0 * np.log(2) * ((x - cx)**2 + (y - cy)**2) / (sigma**2))
    return heatmap.astype(np.float32)


# ── Pre-compute and save all heatmaps to disk ─────────────────────────────
def precompute_heatmaps(dataset_info, heatmap_dir, size, sigma):
    """
    Generate Gaussian heatmaps for every sample and persist as .npy files.
    Skips files that already exist (safe to re-run).
    Returns list of heatmap file paths, one per sample.
    """
    heatmap_paths = []
    print(f"Pre-computing {len(dataset_info):,} heatmaps at {size}×{size} ...")

    for i, item in enumerate(tqdm(dataset_info)):
        out_path = Path(heatmap_dir) / f"heatmap_{i:05d}.npy"

        if not out_path.exists():
            if item['point'] is not None:
                # Scale lesion coordinates from original image space → heatmap space
                cx = item['point'][0] * size / item['width']
                cy = item['point'][1] * size / item['height']
                center_scaled = (cx, cy)
            else:
                center_scaled = None   # Normal → zero heatmap

            hm = generate_gaussian_heatmap(size, center_scaled, sigma)
            np.save(out_path, hm)

        heatmap_paths.append(str(out_path))

    print(f"Done! Heatmaps saved to: {heatmap_dir}")
    return heatmap_paths


heatmap_paths = precompute_heatmaps(
    dataset_info, HEATMAP_DIR, HEATMAP_SIZE, HEATMAP_SIGMA)

# Attach heatmap paths to dataset_info
for item, hp in zip(dataset_info, heatmap_paths):
    item['heatmap_path'] = hp

print(f"\nSample — class: {dataset_info[0]['class']}, "
      f"point: {dataset_info[0]['point']}, "
      f"heatmap max: {np.load(dataset_info[0]['heatmap_path']).max():.4f}")



In [ ]:
# ── Visualise heatmap overlays ────────────────────────────────────────────
sample_idxs = random.sample(range(len(dataset_info)), 12)
fig, axes = plt.subplots(2, 6, figsize=(18, 6))

for ax, idx in zip(axes.flatten(), sample_idxs):
    item = dataset_info[idx]
    img  = cv2.cvtColor(cv2.imread(item['img_path']), cv2.COLOR_BGR2RGB)
    img  = cv2.resize(img, (192, 192))
    hm   = np.load(item['heatmap_path'])
    hm_r = cv2.resize(hm, (192, 192))

    ax.imshow(img)
    ax.imshow(hm_r, cmap='hot', alpha=0.55, vmin=0, vmax=1)
    ax.set_title(item['class'], fontsize=6)
    ax.axis('off')

plt.suptitle('Pre-computed Gaussian Heatmap Overlays
'
             '(Normal classes → blank / all-zero heatmap)', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'heatmap_samples.png', dpi=150)
plt.show()



In [ ]:
# ── Stratified Train / Val Split (80 / 20) ───────────────────────────────
labels_all = [item['class'] for item in dataset_info]

train_info, val_info = train_test_split(
    dataset_info,
    test_size=0.2,
    random_state=SEED,
    stratify=labels_all
)

print(f"Train samples : {len(train_info):,}")
print(f"Val   samples : {len(val_info):,}")

# ── Class weights to handle imbalance across 30 classes ──────────────────
train_labels_int = [class_to_idx[item['class']] for item in train_info]

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_labels_int
)
class_weight_dict = {i: float(w) for i, w in enumerate(class_weights_arr)}

print(f"\nClass weight — max  : {max(class_weight_dict.values()):.3f}")
print(f"Class weight — min  : {min(class_weight_dict.values()):.3f}")
print(f"Class weight — mean : {np.mean(list(class_weight_dict.values())):.3f}")



## 4. 🚀 Pipeline Dữ liệu Tốc độ Cao — `tf.data.Dataset`

### Tại sao KHÔNG dùng `ImageDataGenerator`?
`ImageDataGenerator` chạy trên CPU, gây bottleneck và lãng phí GPU.
`tf.data` với `.cache()` + `.prefetch()` cho phép GPU luôn bận rộn 100%.

### Thứ tự pipeline (quan trọng):
```
from_tensor_slices → map(load) → cache() → shuffle → batch → map(augment) → prefetch(AUTOTUNE)
```
- `cache()` **sau khi load**: tránh đọc lại ảnh từ đĩa mỗi epoch
- Augmentation **sau cache**: mỗi epoch nhận augmentation ngẫu nhiên khác nhau
- `prefetch(AUTOTUNE)`: GPU tính toán batch n trong khi CPU chuẩn bị batch n+1



In [ ]:
# ── Sample weights (từ class weights) ────────────────────────────────────
def make_sample_weights(info_list, class_to_idx, cw_dict):
    return np.array(
        [cw_dict[class_to_idx[item['class']]] for item in info_list],
        dtype=np.float32
    )

train_sw = make_sample_weights(train_info, class_to_idx, class_weight_dict)
val_sw   = np.ones(len(val_info), dtype=np.float32)   # uniform for val


# ── GPU-accelerated augmentation model ────────────────────────────────────
# Applied AFTER cache() → different randomisation each epoch
augmentation_model = keras.Sequential([
    layers.RandomRotation(0.10, seed=SEED),
    layers.RandomTranslation(0.10, 0.10, seed=SEED),
    layers.RandomZoom(0.10, seed=SEED),
    layers.RandomContrast(0.15, seed=SEED),
], name='augmentation')



In [ ]:
# ── Map functions ─────────────────────────────────────────────────────────
def load_sample(img_path, heatmap_path, label, sample_weight):
    """Load image + heatmap from disk. Images kept in [0,255] for EfficientNet."""
    # Load & resize image
    raw  = tf.io.read_file(img_path)
    img  = tf.image.decode_image(raw, channels=3, expand_animations=False)
    img  = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img  = tf.cast(img, tf.float32)          # [0, 255]; EfficientNet handles norm internally

    # Load pre-computed heatmap via numpy (small file, fast)
    heatmap = tf.numpy_function(
        func=lambda p: np.load(p.decode()).astype(np.float32),
        inp=[heatmap_path],
        Tout=tf.float32
    )
    heatmap = tf.reshape(heatmap, [HEATMAP_SIZE, HEATMAP_SIZE, 1])

    return img, {'class_output': label, 'heatmap_output': heatmap}, sample_weight


def augment_batch(img_batch, targets, weights):
    """Apply GPU augmentation layers to a full batch."""
    img_batch = augmentation_model(img_batch, training=True)
    img_batch = tf.clip_by_value(img_batch, 0.0, 255.0)
    return img_batch, targets, weights


# ── Dataset builder ────────────────────────────────────────────────────────
def build_dataset(info_list, sample_weights, augment=False):
    img_paths     = [item['img_path']     for item in info_list]
    heatmap_paths = [item['heatmap_path'] for item in info_list]
    labels        = np.array([
        keras.utils.to_categorical(class_to_idx[item['class']], NUM_CLASSES)
        for item in info_list
    ], dtype=np.float32)

    ds = tf.data.Dataset.from_tensor_slices(
        (img_paths, heatmap_paths, labels, sample_weights))

    # Load samples in parallel
    ds = ds.map(load_sample, num_parallel_calls=tf.data.AUTOTUNE)

    # Cache loaded data → avoid disk I/O on subsequent epochs
    ds = ds.cache()

    # Shuffle only for training (after cache for proper per-epoch shuffling)
    if augment:
        ds = ds.shuffle(
            buffer_size=min(len(info_list), 4096),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    # Batch
    ds = ds.batch(BATCH_SIZE, drop_remainder=augment)

    # GPU augmentation on full batches (after cache, fresh each epoch)
    if augment:
        ds = ds.map(augment_batch, num_parallel_calls=tf.data.AUTOTUNE)

    # Keep GPU fed asynchronously
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


print("Building datasets ...")
train_ds = build_dataset(train_info, train_sw, augment=True)
val_ds   = build_dataset(val_info,   val_sw,   augment=False)
print("Train dataset:", train_ds)
print("Val   dataset:", val_ds)



In [ ]:
# ── Sanity check: one batch ───────────────────────────────────────────────
for X_b, Y_b, W_b in train_ds.take(1):
    print(f"Image  shape : {X_b.shape}  dtype={X_b.dtype}")
    print(f"Label  shape : {Y_b['class_output'].shape}")
    print(f"Heatmap shape: {Y_b['heatmap_output'].shape}")
    print(f"Weight shape : {W_b.shape}")
    print(f"Pixel range  : [{X_b.numpy().min():.1f}, {X_b.numpy().max():.1f}]")

# Visualise
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for i in range(8):
    img = X_b[i].numpy().astype(np.uint8)
    hm  = Y_b['heatmap_output'][i, :, :, 0].numpy()
    cls_idx = np.argmax(Y_b['class_output'][i].numpy())

    axes[0, i].imshow(img)
    axes[0, i].set_title(CLASS_NAMES[cls_idx], fontsize=6)
    axes[0, i].axis('off')

    axes[1, i].imshow(hm, cmap='hot', vmin=0, vmax=1)
    axes[1, i].set_title('Target Heatmap', fontsize=6)
    axes[1, i].axis('off')

plt.suptitle('Augmented Training Batch (Row 1: Image | Row 2: Target Heatmap)', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'training_batch_sample.png', dpi=150)
plt.show()



## 5. 🏗️ Kiến trúc Mô hình: Dual-Head EfficientNetV2S

```
Input (384×384×3)
    └── EfficientNetV2S backbone (ImageNet, include_preprocessing=True)
         │   backbone output: 12×12×1280
         │
         ├── HEAD 1 – Classification
         │       GlobalAveragePooling2D
         │       Dropout(0.4)
         │       Dense(30, softmax, dtype=float32)     →  class_output
         │
         └── HEAD 2 – Heatmap / Attention Decoder
                 Conv2DTranspose(128, stride=2)   12×12 → 24×24
                 Conv2DTranspose(64,  stride=2)   24×24 → 48×48
                 Conv2DTranspose(1,   stride=1, sigmoid, dtype=float32)  →  heatmap_output
```

**Loss configuration:**
```
loss_weights = {'class_output': 1.0, 'heatmap_output': 0.1}
```
Heatmap loss = 10% weight → nhẹ nhàng hướng dẫn attention, không làm loãng mục tiêu chính.



In [ ]:
def build_dual_head_model(img_size, heatmap_size, num_classes):
    """
    Dual-head EfficientNetV2S.
    - Input  : (img_size, img_size, 3) in [0, 255] range
    - Output 1 (class_output)  : (num_classes,) softmax
    - Output 2 (heatmap_output): (heatmap_size, heatmap_size, 1) sigmoid
    """
    inputs = keras.Input(shape=(img_size, img_size, 3), name='input_image')

    # ── Backbone ──────────────────────────────────────────────────────────
    # include_preprocessing=True: EfficientNetV2S normalises [0,255] → [-1,1] internally
    backbone = EfficientNetV2S(
        include_top=False,
        weights='imagenet',
        input_tensor=inputs,
        include_preprocessing=True
    )
    backbone.trainable = False    # frozen during Phase 1

    x = backbone.output           # (batch, 12, 12, 1280) for 384×384 input

    # ── Head 1: Classification ────────────────────────────────────────────
    gap = layers.GlobalAveragePooling2D(name='gap')(x)
    gap = layers.Dropout(0.4, name='drop_cls')(gap)
    class_out = layers.Dense(
        num_classes,
        activation='softmax',
        dtype='float32',              # force float32 — mixed-precision stability
        name='class_output'
    )(gap)

    # ── Head 2: Heatmap Decoder (12×12 → 24×24 → 48×48) ─────────────────
    h = layers.Conv2DTranspose(
        128, kernel_size=3, strides=2, padding='same',
        activation='relu', name='up_1'
    )(x)                              # 24×24×128
    h = layers.Conv2DTranspose(
        64,  kernel_size=3, strides=2, padding='same',
        activation='relu', name='up_2'
    )(h)                              # 48×48×64
    heatmap_out = layers.Conv2DTranspose(
        1, kernel_size=1, strides=1, padding='same',
        activation='sigmoid',
        dtype='float32',              # force float32
        name='heatmap_output'
    )(h)                              # 48×48×1

    model = keras.Model(
        inputs=inputs,
        outputs=[class_out, heatmap_out],
        name='BrainTumor_DualHead_EfficientNetV2S'
    )
    return model, backbone


model, backbone = build_dual_head_model(IMG_SIZE, HEATMAP_SIZE, NUM_CLASSES)
model.summary(line_length=110, expand_nested=False)

total      = model.count_params()
trainable  = sum(tf.size(v).numpy() for v in model.trainable_variables)
print(f"\nTotal params     : {total:,}")
print(f"Trainable params : {trainable:,}  (heads only — backbone frozen)")



In [ ]:
# ── Compile helper (reused in both phases) ───────────────────────────────
METRICS_CLS = [
    'accuracy',
    keras.metrics.Precision(name='precision'),
    keras.metrics.Recall(name='recall'),
    keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc'),
]

def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss={
            'class_output'  : 'categorical_crossentropy',
            'heatmap_output': 'mean_squared_error',
        },
        loss_weights={
            'class_output'  : 1.0,    # primary task — full weight
            'heatmap_output': 0.1,    # attention guide — light regularisation
        },
        metrics={
            'class_output'  : METRICS_CLS,
            'heatmap_output': [],
        }
    )

compile_model(model, lr=1e-3)
print("Model compiled.")



## 6. 🔥 Phase 1 — Warm-up Training (Backbone Đóng Băng)

**Mục tiêu:** Train nhanh 2 custom heads trong khi backbone giữ nguyên trọng số ImageNet.
- Learning rate cao (1e-3) → heads học nhanh
- 10 epochs đủ để heads khởi động, tránh gradient shock khi fine-tune
- Sample weights áp dụng trực tiếp trong dataset → xử lý class imbalance



In [ ]:
BEST_P1_PATH = MODEL_DIR / 'brain_tumor_phase1_best.keras'
BEST_FT_PATH = MODEL_DIR / 'brain_tumor_finetuned_best.keras'
LOG_P1_PATH  = LOG_DIR   / 'phase1_log.csv'
LOG_P2_PATH  = LOG_DIR   / 'phase2_log.csv'

callbacks_p1 = [
    ModelCheckpoint(
        filepath=BEST_P1_PATH,
        monitor='val_class_output_accuracy',
        save_best_only=True, mode='max', verbose=1
    ),
    EarlyStopping(
        monitor='val_class_output_accuracy',
        patience=5, restore_best_weights=True, mode='max', verbose=1
    ),
    CSVLogger(filename=LOG_P1_PATH, append=False),
]

print("=== PHASE 1: Warm-up — backbone frozen, training heads only ===")
history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_P1,
    callbacks=callbacks_p1,
    verbose=1
)

p1_epochs = len(history_p1.history['class_output_accuracy'])
p1_best_acc = max(history_p1.history['val_class_output_accuracy'])
print(f"\nPhase 1 — Epochs run: {p1_epochs}")
print(f"Phase 1 — Best val accuracy: {p1_best_acc:.4f}")



## 7. 🎯 Phase 2 — Fine-tuning (Mở Băng Top Backbone Layers)

**Chiến lược:**
1. Load best checkpoint từ Phase 1
2. Mở **70 lớp cuối** của backbone (giữ BatchNorm ở inference mode để tránh instability)
3. **CosineDecayRestarts**: LR mượt từ `2e-4` → `1e-6` với warm restarts → thoát local minima
4. EarlyStopping patience=10 → cho đủ thời gian hội tụ sau mỗi restart



In [ ]:
# Restore best Phase-1 weights
model.load_weights(BEST_P1_PATH)
print(f"Loaded Phase 1 best weights from: {BEST_P1_PATH}")

# ── Unfreeze top 70 backbone layers ──────────────────────────────────────
backbone.trainable = True
FINE_TUNE_AT = len(backbone.layers) - 70

for layer in backbone.layers[:FINE_TUNE_AT]:
    layer.trainable = False

for layer in backbone.layers[FINE_TUNE_AT:]:
    # BatchNorm stays in inference mode — critical for fine-tune stability
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = True

trainable_cnt = sum(1 for l in backbone.layers if l.trainable)
print(f"Trainable backbone layers: {trainable_cnt} / {len(backbone.layers)}")

# ── Cosine Decay with Warm Restarts ──────────────────────────────────────
steps_per_epoch = len(train_ds)
total_steps     = steps_per_epoch * EPOCHS_P2
first_decay     = total_steps // 3          # first restart at 1/3 of total

lr_schedule = keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=2e-4,
    first_decay_steps=first_decay,
    t_mul=1.0,                              # equal restart intervals
    m_mul=0.9,                              # LR decays 10% each restart
    alpha=1e-6                              # minimum LR floor
)

compile_model(model, lr=lr_schedule)

callbacks_p2 = [
    ModelCheckpoint(
        filepath=BEST_FT_PATH,
        monitor='val_class_output_accuracy',
        save_best_only=True, mode='max', verbose=1
    ),
    EarlyStopping(
        monitor='val_class_output_accuracy',
        patience=10, restore_best_weights=True, mode='max', verbose=1
    ),
    CSVLogger(filename=LOG_P2_PATH, append=False),
]

print("\n=== PHASE 2: Fine-tuning top 70 backbone layers ===")
history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_P2,
    callbacks=callbacks_p2,
    verbose=1
)

p2_epochs   = len(history_p2.history['class_output_accuracy'])
p2_best_acc = max(history_p2.history['val_class_output_accuracy'])
print(f"\nPhase 2 — Epochs run : {p2_epochs}")
print(f"Phase 2 — Best val acc: {p2_best_acc:.4f}  ({p2_best_acc*100:.2f}%)")
print(f"Model saved to: {BEST_FT_PATH}")



## 8. 📈 Biểu đồ Quá trình Huấn luyện (Phase 1 + Phase 2)

In [ ]:
def plot_combined(h1, h2, metric, val_metric, title, ylabel, save_name, phase_sep):
    """Merge Phase 1 + Phase 2 history and plot with phase separator."""
    train_v = h1.history.get(metric, [])     + h2.history.get(metric, [])
    val_v   = h1.history.get(val_metric, []) + h2.history.get(val_metric, [])
    epochs  = range(1, len(train_v) + 1)
    ymax    = max(max(train_v), max(val_v)) * 1.05

    plt.figure(figsize=(11, 5))
    plt.plot(epochs, train_v, label='Train',      linewidth=2, color='steelblue')
    plt.plot(epochs, val_v,   label='Validation', linewidth=2, color='darkorange')
    plt.axvline(phase_sep, color='gray', linestyle='--', linewidth=1.5,
                label=f'Phase 1 → 2  (epoch {phase_sep})')
    plt.fill_betweenx([0, ymax], 0, phase_sep,
                      alpha=0.07, color='blue',  label='Phase 1 (frozen)')
    plt.fill_betweenx([0, ymax], phase_sep, len(train_v),
                      alpha=0.07, color='green', label='Phase 2 (fine-tune)')
    plt.title(title, fontsize=14)
    plt.xlabel('Epoch'); plt.ylabel(ylabel)
    plt.legend(loc='lower right'); plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / save_name, dpi=150)
    plt.show()


phase_sep = len(history_p1.history['class_output_accuracy'])

plot_combined(history_p1, history_p2,
    'class_output_accuracy',     'val_class_output_accuracy',
    'Model Accuracy Curve',      'Accuracy',
    'accuracy_curve.png',        phase_sep)

plot_combined(history_p1, history_p2,
    'class_output_loss',         'val_class_output_loss',
    'Classification Loss Curve', 'Loss',
    'loss_curve.png',            phase_sep)

plot_combined(history_p1, history_p2,
    'class_output_recall',       'val_class_output_recall',
    'Recall Curve',              'Recall',
    'recall_curve.png',          phase_sep)



## 9. 📊 Đánh giá Mô hình & Trực quan hóa Kết quả

Load best weights từ Phase 2, chạy toàn bộ validation set.
- **Classification Report**: per-class precision, recall, F1
- **Normalised Confusion Matrix**: dễ thấy nhầm lẫn giữa các class
- **3×3 Prediction Grid**: original image + predicted heatmap overlay + heatmap riêng



In [ ]:
# Load best fine-tuned model
best_model = keras.models.load_model(BEST_FT_PATH)
print(f"Loaded best model : {BEST_FT_PATH}")



In [ ]:
# ── Run inference over full validation set ───────────────────────────────
print("Running inference on validation set ...")
cls_probs_all, heatmaps_all, y_true_all, raw_imgs_all = [], [], [], []

for X_batch, Y_batch, _ in val_ds:
    preds = best_model(X_batch, training=False)
    cls_probs_all.append(preds[0].numpy())
    heatmaps_all.append(preds[1].numpy())
    y_true_all.append(np.argmax(Y_batch['class_output'].numpy(), axis=1))
    raw_imgs_all.append(X_batch.numpy().astype(np.uint8))

y_pred_prob = np.concatenate(cls_probs_all,  axis=0)
y_heatmaps  = np.concatenate(heatmaps_all,   axis=0)
y_pred      = np.argmax(y_pred_prob, axis=1)
y_true      = np.concatenate(y_true_all, axis=0)
raw_imgs    = np.concatenate(raw_imgs_all, axis=0)

# Trim to actual val size (drop_remainder may have dropped a partial batch)
n = min(len(y_true), len(val_info))
y_pred, y_true, y_heatmaps, raw_imgs = (
    y_pred[:n], y_true[:n], y_heatmaps[:n], raw_imgs[:n])
print(f"Evaluated samples: {n:,}")



In [ ]:
# ── Scalar metrics ────────────────────────────────────────────────────────
acc       = accuracy_score(y_true, y_pred)
f1_macro  = f1_score(y_true, y_pred, average='macro',    zero_division=0)
f1_weight = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print(f"Validation Accuracy      : {acc:.4f}  ({acc*100:.2f}%)")
print(f"Macro  F1-score          : {f1_macro:.4f}")
print(f"Weighted F1-score        : {f1_weight:.4f}")
print("\n===== Per-Class Classification Report =====")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

# Save
metrics_df = pd.DataFrame([{
    'model'       : 'EfficientNetV2S DualHead (fine-tuned)',
    'img_size'    : IMG_SIZE,
    'heatmap_size': HEATMAP_SIZE,
    'epochs_p1'   : phase_sep,
    'epochs_p2'   : len(history_p2.history['class_output_loss']),
    'val_accuracy': float(acc),
    'f1_macro'    : float(f1_macro),
    'f1_weighted' : float(f1_weight),
}])
display(metrics_df)
metrics_df.to_csv(REPORT_DIR / 'metrics_summary.csv', index=False)



In [ ]:
# ── Normalised Confusion Matrix ───────────────────────────────────────────
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(28, 12))
for ax, data, fmt, title in zip(
    axes, [cm, cm_norm], ['d', '.2f'],
    ['Confusion Matrix (counts)', 'Confusion Matrix (normalised)']):

    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, annot_kws={'size': 6})
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Predicted Class', fontsize=10)
    ax.set_ylabel('True Class',      fontsize=10)
    ax.tick_params(axis='x', rotation=90, labelsize=6)
    ax.tick_params(axis='y', rotation=0,  labelsize=6)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'confusion_matrix.png', dpi=150)
plt.show()



In [ ]:
# ── 3×3 Prediction Grid: original | heatmap overlay | heatmap alone ──────
n_grid = 9
indices = list(range(n_grid))   # first 9 validation samples

fig, axes = plt.subplots(3, n_grid, figsize=(24, 8))
row_labels = ['Original Image', 'Heatmap Overlay', 'Predicted Heatmap']

for col, idx in enumerate(indices):
    img     = raw_imgs[idx]
    hm      = y_heatmaps[idx, :, :, 0]
    hm_big  = cv2.resize(hm, (IMG_SIZE, IMG_SIZE))
    pred_cls = CLASS_NAMES[y_pred[idx]]
    true_cls = CLASS_NAMES[y_true[idx]]
    correct  = y_pred[idx] == y_true[idx]
    border   = 'green' if correct else 'red'

    # Row 0: original image + true label
    axes[0, col].imshow(img)
    axes[0, col].set_title(f'True:\n{true_cls}', fontsize=6)
    axes[0, col].axis('off')
    for spine in axes[0, col].spines.values():
        spine.set_edgecolor(border); spine.set_linewidth(3)
    axes[0, col].set_frame_on(True)

    # Row 1: original + heatmap overlay + predicted label
    axes[1, col].imshow(img)
    axes[1, col].imshow(hm_big, cmap='jet', alpha=0.50, vmin=0, vmax=1)
    axes[1, col].set_title(f'Pred:\n{pred_cls}', fontsize=6, color=border)
    axes[1, col].axis('off')

    # Row 2: heatmap alone
    axes[2, col].imshow(hm_big, cmap='hot', vmin=0, vmax=1)
    axes[2, col].set_title('Attention Map', fontsize=6)
    axes[2, col].axis('off')

# Row labels
for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=9, rotation=90, labelpad=40)

plt.suptitle(
    'Validation Predictions  |  Green border = Correct  |  Red border = Wrong',
    fontsize=11)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'prediction_grid.png', dpi=150)
plt.show()



In [ ]:
print("=" * 65)
print("  TRAINING & EVALUATION COMPLETE")
print("=" * 65)
print(f"  Best model        : {BEST_FT_PATH}")
print(f"  Validation Acc    : {acc*100:.2f}%")
print(f"  Macro F1-score    : {f1_macro:.4f}")
print(f"  Weighted F1-score : {f1_weight:.4f}")
print(f"  Figures saved to  : {FIGURE_DIR}")
print(f"  Logs saved to     : {LOG_DIR}")
print("=" * 65)

